# 05 - HSV + GrabCut + Cleaning Segmentation

Notebook ini memakai pipeline segmentasi baru tanpa mengubah `04_segmentation.ipynb` dan tanpa mengubah `src/segmentation.py`.

Pipeline utama:

1. HSV membuat mask awal foreground.
2. GrabCut memperbaiki batas objek berdasarkan mask awal.
3. Cleaning morfologi mengurangi noise dan lubang kecil.
4. Mask final dipakai untuk mengambil objek dari gambar asli.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

project_root = Path("..").resolve()
sys.path.append(str(project_root / "src"))

from hsv_grabcut_segmentation import (
    hsv_grabcut_pipeline,
    foreground_ratio,
    batch_save,
)

plt.rcParams["figure.figsize"] = (12, 6)

## Ambil Sample Gambar

In [ ]:
image_dir = project_root / "data" / "enhanced" / "clahe" / "val"
image_paths = sorted(image_dir.rglob("*.jpg"))

print("Jumlah gambar:", len(image_paths))
sample_path = image_paths[0]
print("Sample:", sample_path)

img = cv2.imread(str(sample_path))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()

## Jalankan Pipeline HSV + GrabCut + Cleaning

In [ ]:
init, refined, final, segmented = hsv_grabcut_pipeline(
    img,
    return_steps=True,
    keep_largest=False,
)

print("Foreground ratio:", round(foreground_ratio(final), 3))

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(20, 5))

ax[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
ax[0].set_title("Original")

ax[1].imshow(init, cmap="gray")
ax[1].set_title("HSV Init")

ax[2].imshow(refined, cmap="gray")
ax[2].set_title("GrabCut Refined")

ax[3].imshow(final, cmap="gray")
ax[3].set_title("Final Mask")

ax[4].imshow(cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB))
ax[4].set_title("Segmented")

for axis in ax:
    axis.axis("off")

plt.tight_layout()
plt.show()

## Uji Beberapa Sampel Acak

In [ ]:
rng = np.random.default_rng(42)
sample_count = min(5, len(image_paths))
sample_indices = rng.choice(len(image_paths), size=sample_count, replace=False)
samples = [image_paths[i] for i in sample_indices]

fig, ax = plt.subplots(sample_count, 3, figsize=(12, 4 * sample_count))

if sample_count == 1:
    ax = np.expand_dims(ax, axis=0)

for i, img_path in enumerate(samples):
    image = cv2.imread(str(img_path))
    _, _, mask, segmented_image = hsv_grabcut_pipeline(
        image,
        return_steps=True,
        keep_largest=False,
    )

    ax[i, 0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax[i, 0].set_title(img_path.parent.name)

    ax[i, 1].imshow(mask, cmap="gray")
    ax[i, 1].set_title(f"Mask | ratio={foreground_ratio(mask):.2f}")

    ax[i, 2].imshow(cv2.cvtColor(segmented_image, cv2.COLOR_BGR2RGB))
    ax[i, 2].set_title("Segmented")

    for j in range(3):
        ax[i, j].axis("off")

plt.tight_layout()
plt.show()

## Statistik Foreground Ratio

Nilai ekstrem mendekati 0 atau 1 perlu dicek secara visual karena biasanya menandakan segmentasi gagal.

In [ ]:
ratios = []
problem_images = []

for img_path in image_paths:
    image = cv2.imread(str(img_path))
    if image is None:
        continue

    mask, _ = hsv_grabcut_pipeline(image, keep_largest=False)
    ratio = foreground_ratio(mask)
    ratios.append(ratio)

    if ratio < 0.02 or ratio > 0.95:
        problem_images.append((img_path, ratio))

ratios = np.array(ratios)

print("Mean:", round(float(ratios.mean()), 3))
print("Min :", round(float(ratios.min()), 3))
print("Max :", round(float(ratios.max()), 3))
print("Problem images:", len(problem_images))

for path, ratio in problem_images[:10]:
    print(round(ratio, 3), path)

## Simpan Hasil Segmentasi

Cell ini menyimpan mask dan gambar hasil segmentasi ke folder `results/segmentation/hsv_grabcut_cleaning`.

In [ ]:
# Jalankan cell ini kalau ingin menyimpan semua hasil untuk split tertentu.
# output_dir = batch_save(enhancement="clahe", split="val", keep_largest=False)
# print("Saved to:", output_dir)